In [0]:
from datetime import datetime
import uuid

run_id = "uc16_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")
task_name = "build_appointments_clean"

spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'STARTED',
  current_timestamp(),
  NULL,
  NULL,
  NULL
)
""")

In [0]:
%sql
CREATE OR REPLACE TABLE poc_health_no_show.gold.outreach_prompt_queue AS
SELECT *
FROM poc_health_no_show.gold.appointment_risk_segments
WHERE risk_segment IN ('HIGH', 'MEDIUM_HIGH');

In [0]:
%sql
CREATE OR REPLACE TABLE poc_health_no_show.gold.outreach_prompt_queue
COMMENT 'High and medium-high risk appointments selected for AI-generated reminder drafting.'
AS
SELECT
  appointment_id,
  patient_pseudo_id,
  appointment_date,
  appointment_weekday,
  risk_segment,
  risk_score,
  risk_reason,
  recommended_action
FROM poc_health_no_show.gold.appointment_risk_segments
WHERE risk_segment IN ('HIGH', 'MEDIUM_HIGH');

In [0]:
%sql
CREATE OR REPLACE TABLE poc_health_no_show.gold.outreach_prompt_sample
COMMENT 'Limited sample used for AI outreach drafting to control cost and runtime in the POC.'
AS
SELECT *
FROM poc_health_no_show.gold.outreach_prompt_queue
ORDER BY risk_score DESC, appointment_date
LIMIT 100;

In [0]:
spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'SUCCESS',
  current_timestamp(),
  current_timestamp(),
  NULL,
  NULL
)
""")